# Ders 2: Ölçekleme Yasaları ve Modern Mimariler

**İleri Derin Öğrenme** — Haydar Kılıç

Ön koşul: *Derin Öğrenme*, Ders 6–7 (ESA'lar, Transformer'lar).

Model ailesi sabitlendiğinde derin öğrenme şaşırtıcı ölçüde öngörülebilir hâle gelir: kayıp,
parametre, veri ve hesap gücünün bir **kuvvet yasası** olarak düşer. Bu defterde hesap-optimal
dağılım kuralını türetiyor, "ortaya çıkan yetenekler"in (emergence) neden kısmen bir ölçüm yanılsaması
olduğunu gösteriyor ve bir transformer'da parametrelerle FLOP'ların gerçekte nerede durduğuna
bakıyoruz — ki Mixture-of-Experts'in çıkış noktası tam olarak budur.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
plt.rcParams["figure.dpi"] = 100
print("Kütüphaneler yüklendi.")


## 1. Kuvvet Yasaları

Transformer dil modelleri için deneysel olarak

$$L(N) = \left(\frac{N_c}{N}\right)^{\alpha_N}, \qquad
  L(D) = \left(\frac{D_c}{D}\right)^{\alpha_D},$$

burada $N$ parametre, $D$ eğitim token sayısıdır. Üsler küçüktür ($\alpha \approx 0.05$–$0.35$) ve
işin can sıkıcı tarafı da budur: $\alpha = 0.076$ olan bir kuvvet yasasında kaybı bir kat büyüklük
düşürmek için $N$'yi kabaca $10^{13}$ katına çıkarmak gerekir.

Log–log grafikte kuvvet yasası bir doğrudur; dolayısıyla uydurma işlemi log uzayında sıradan en
küçük karelerdir.


In [ ]:
# Sentetik "deney": birkaç model boyutunda kaybı ölç, sonra kuvvet yasası uydur.
alpha_true, Nc_true, L_inf = 0.076, 8.8e13, 1.69     # indirgenemez kayıp L_inf (verinin entropisi)
N = np.logspace(6, 11, 12)
L_obs = L_inf + (Nc_true/N)**alpha_true * np.exp(np.random.normal(0, 0.01, N.size))

# Uydurma:  log(L - L_inf) = alpha*log(Nc) - alpha*log(N)
y = np.log(L_obs - L_inf)
A = np.vstack([np.ones_like(N), -np.log(N)]).T
coef, *_ = np.linalg.lstsq(A, y, rcond=None)
alpha_fit = coef[1]
Nc_fit    = np.exp(coef[0]/alpha_fit)
print(f"gerçek alpha = {alpha_true:.4f}   uydurulan alpha = {alpha_fit:.4f}")
print(f"gerçek N_c   = {Nc_true:.2e}      uydurulan N_c   = {Nc_fit:.2e}")

Ng = np.logspace(6, 13, 200)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
axes[0].loglog(N, L_obs, "o", ms=6, label="ölçülen koşular")
axes[0].loglog(Ng, L_inf + (Nc_fit/Ng)**alpha_fit, lw=2, label="uydurulan kuvvet yasası")
axes[0].axhline(L_inf, ls="--", c="crimson", lw=1.2, label=f"indirgenemez kayıp = {L_inf}")
axes[0].set_xlabel("parametre sayısı N"); axes[0].set_ylabel("kayıp")
axes[0].set_title("Kayıp - model boyutu"); axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3, which="both")

axes[1].loglog(Ng, (Nc_fit/Ng)**alpha_fit, lw=2, c="seagreen")
axes[1].set_xlabel("parametre sayısı N"); axes[1].set_ylabel("indirgenebilir kayıp  L - L_inf")
axes[1].set_title("L_inf çıkarılınca kuvvet yasası bir doğruya dönüşür")
axes[1].grid(alpha=0.3, which="both")

plt.tight_layout(); plt.show()

gain = (10**1)**(1/alpha_fit)
print(f"\nİndirgenebilir kaybı 10'a bölmek için N'yi {gain:.2e} katına çıkarmak gerekir.")


## 2. Hesap-Optimal Ölçekleme ("Chinchilla" Hesabı)

Sabit bir hesap bütçesiyle: büyük bir modeli az veriyle mi, küçük bir modeli çok veriyle mi eğitmeli?
Kaybı ayrı terimlerle yazıp standart transformer FLOP kestirimi $C = 6ND$'yi kullanalım:

$$L(N, D) = E + \frac{A}{N^{\alpha}} + \frac{B}{D^{\beta}}, \qquad C \approx 6ND.$$

$D = C/(6N)$ yerine konup $\partial L/\partial N = 0$ yapılırsa

$$\alpha A N^{-\alpha-1} = \beta B \left(\frac{C}{6}\right)^{-\beta} N^{\beta-1}
\;\Longrightarrow\; N^{\star} \propto C^{\frac{\beta}{\alpha+\beta}}, \quad
D^{\star} \propto C^{\frac{\alpha}{\alpha+\beta}} .$$

Yayımlanmış uydurmalarla ($\alpha \approx 0.34$, $\beta \approx 0.28$) iki üs yaklaşık $0.45$ ve
$0.55$ çıkar: **parametre ile token sayısı kabaca aynı hızda büyümelidir**. Bu, $N$'yi $D$'den çok
daha hızlı büyüten ve dolayısıyla büyük modelleri fazlasıyla az veriyle eğiten önceki pratiğe
yapılmış bir düzeltmeydi.


In [ ]:
E, A, B, alpha, beta = 1.69, 406.4, 410.7, 0.34, 0.28     # Hoffmann vd. tarzı katsayılar
Lnd = lambda N, D: E + A/N**alpha + B/D**beta

def optimal_split(C, n_grid=4000):
    N = np.logspace(7, 13, n_grid)
    D = C/(6*N)
    L = Lnd(N, D)
    i = int(np.argmin(L))
    return N[i], D[i], L[i]

budgets = np.logspace(18, 25, 40)
res = np.array([optimal_split(c) for c in budgets])
Nstar, Dstar, Lstar = res[:, 0], res[:, 1], res[:, 2]

# Empirical exponents of the optimal frontier
aN = np.polyfit(np.log(budgets), np.log(Nstar), 1)[0]
aD = np.polyfit(np.log(budgets), np.log(Dstar), 1)[0]
print(f"N* ~ C^{aN:.3f}     D* ~ C^{aD:.3f}     (teori: beta/(a+b)={beta/(alpha+beta):.3f}, "
      f"alpha/(a+b)={alpha/(alpha+beta):.3f})")
print(f"C=1e23'te optimal token/parametre oranı: {Dstar[np.argmin(abs(budgets-1e23))]/Nstar[np.argmin(abs(budgets-1e23))]:.0f}")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
C_fixed = 1e22
Ns = np.logspace(8, 12, 300)
axes[0].semilogx(Ns, Lnd(Ns, C_fixed/(6*Ns)), lw=2)
n0, d0, l0 = optimal_split(C_fixed)
axes[0].scatter([n0], [l0], s=80, c="crimson", zorder=4, label=f"optimum N={n0:.1e}")
axes[0].set_xlabel("parametre sayısı N"); axes[0].set_ylabel("kayıp")
axes[0].set_title(f"Sabit hesap C={C_fixed:.0e}: iç bir optimum")
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3, which="both")

axes[1].loglog(budgets, Nstar, lw=2, label="N* (parametre)")
axes[1].loglog(budgets, Dstar, lw=2, label="D* (token)")
axes[1].set_xlabel("hesap C (FLOP)"); axes[1].set_title("Hesap-optimal sınır")
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3, which="both")

# Kayıp yüzeyinin ısı haritası ve üzerine çizilmiş optimal yol
Ngrid = np.logspace(8, 12, 160); Dgrid = np.logspace(9, 13, 160)
NN, DD = np.meshgrid(Ngrid, Dgrid)
im = axes[2].pcolormesh(NN, DD, Lnd(NN, DD), shading="auto", cmap="viridis")
axes[2].plot(Nstar, Dstar, c="crimson", lw=2.5, label="hesap-optimal yol")
for c in [1e21, 1e22, 1e23]:
    axes[2].plot(Ngrid, c/(6*Ngrid), ls="--", c="w", lw=0.9)
axes[2].set_xscale("log"); axes[2].set_yscale("log")
axes[2].set_xlim(Ngrid[0], Ngrid[-1]); axes[2].set_ylim(Dgrid[0], Dgrid[-1])
axes[2].set_xlabel("parametre sayısı N"); axes[2].set_ylabel("token D")
axes[2].set_title("Kayıp yüzeyi, eş-hesap eğrileri, optimal yol")
plt.colorbar(im, ax=axes[2], label="kayıp"); axes[2].legend(fontsize=8, loc="lower left")

plt.tight_layout(); plt.show()


Ortadaki panelin *söylemediği* şeye dikkat: hesap-optimal model *eğitmesi* en ucuz olandır,
*servis etmesi* değil. Milyarlarca sorguya cevap verecek bir model için, onu hesap-optimal token
sayısının çok ötesinde eğitmek rasyoneldir; çünkü o noktada toplam bütçeye eğitim değil çıkarım
(inference) maliyeti hâkim olur.

## 3. "Ortaya Çıkış" (Emergence) ve Metrik Seçimi

Bazı yetenekler ölçekle birlikte aniden beliriyormuş gibi görünür. Bu aniliğin önemli bir kısmı
metrikten kaynaklanır. Bir görevin $k$ token'ın tamamının doğru olmasını gerektirdiğini ve token
başına doğruluğun $p(N)$ ölçekle *düzgün* iyileştiğini varsayalım. Tam eşleşme doğruluğu
$p(N)^k$'dir — düzgün bir eğrinin yüksek bir kuvveti eşik gibi görünür. Aynı koşular sürekli
metriklerle (log-olabilirlik, token düzenleme mesafesi) çizildiğinde hiçbir süreksizlik görülmez.


In [ ]:
Nx = np.logspace(6, 11, 300)
p  = 1 - (Nx/1e6)**(-0.12)*0.55            # token başına düzgün değişen doğruluk
p  = np.clip(p, 0, 0.999)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].semilogx(Nx, p, lw=2, c="steelblue")
axes[0].set_title("Token başına doğruluk: tamamen düzgün")
axes[0].set_xlabel("parametre sayısı N"); axes[0].set_ylabel("p(N)"); axes[0].grid(alpha=0.3)

for k, c in zip([1, 5, 20, 50], ["#c6dbef", "#6baed6", "#2171b5", "#08306b"]):
    axes[1].semilogx(Nx, p**k, lw=2, c=c, label=f"tam eşleşme, k={k} token")
axes[1].set_title('Süreksiz bir metrikten doğan "ortaya çıkış"')
axes[1].set_xlabel("parametre sayısı N"); axes[1].set_ylabel("doğruluk"); axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

axes[2].semilogx(Nx, -np.log(p), lw=2, c="seagreen")
axes[2].set_yscale("log")
axes[2].set_title("Aynı koşular, sürekli metrik (token başına NLL)")
axes[2].set_xlabel("parametre sayısı N"); axes[2].set_ylabel("-log p"); axes[2].grid(alpha=0.3, which="both")

plt.tight_layout(); plt.show()
print("Ortadaki paneldeki 'faz geçişi' tamamen metrik tarafından üretiliyor.")


Bu, hiçbir yeteneğin gerçekten süreksiz olmadığını kanıtlamaz — ispat yükünün iddiayı öne sürende
olduğunu ve sürekli bir metriğin daima birlikte raporlanması gerektiğini söyler.

## 4. Parametreler ve FLOP'lar Nerede?

$L$ katmanlı, model genişliği $d$, ileri besleme genişliği $d_{ff} = 4d$ olan yalnızca-kod-çözücü bir
transformer için:

| Bileşen | Katman başına parametre |
|---|---|
| Dikkat $W_Q, W_K, W_V, W_O$ | $4d^2$ |
| İleri besleme $W_1, W_2$ | $2 d \cdot d_{ff} = 8d^2$ |
| **Toplam** | $\approx 12 d^2 L$ |

Yani gömme dışındaki parametrelerin kabaca **üçte ikisi** FFN'de durur. Dikkat FLOP'ları $O(n^2 d)$,
FFN FLOP'ları $O(n d^2)$ ile ölçeklenir; dolayısıyla dikkat ancak dizi uzunluğu $n$ genişlik $d$ ile
kıyaslanabilir hâle geldiğinde baskın olur — kısa dizilerde transformer esasen bir MLP yığınıdır.


In [ ]:
def transformer_params(d, L, vocab=50000, n_ctx=2048, d_ff_mult=4):
    attn  = 4*d*d*L
    ffn   = 2*d*(d_ff_mult*d)*L
    emb   = vocab*d + n_ctx*d
    return dict(attention=attn, ffn=ffn, embedding=emb, total=attn+ffn+emb)

for name, d, L in [("küçük",  768, 12), ("orta", 1600, 48), ("büyük", 4096, 64)]:
    p = transformer_params(d, L)
    print(f"{name:7s} d={d:5d} L={L:3d} | toplam {p['total']/1e9:6.2f}B | "
          f"attn {p['attention']/p['total']:.0%}  ffn {p['ffn']/p['total']:.0%}  emb {p['embedding']/p['total']:.0%}")

d, L = 1600, 48
n = np.logspace(1, 5, 200)
flops_attn = 4*n**2*d*L          # QK^T ve (dikkat)V
flops_ffn  = 16*n*d**2*L         # iki FFN matris çarpımı, d_ff = 4d
flops_proj = 8*n*d**2*L          # Q,K,V,O izdüşümleri

plt.figure(figsize=(8.5, 4.2))
plt.loglog(n, flops_attn, lw=2, label="dikkat  O(n^2 d)")
plt.loglog(n, flops_ffn + flops_proj, lw=2, label="FFN + izdüşümler  O(n d^2)")
plt.loglog(n, flops_attn + flops_ffn + flops_proj, lw=2, ls="--", c="k", label="toplam")
cross = n[np.argmin(np.abs(flops_attn - (flops_ffn + flops_proj)))]
plt.axvline(cross, c="crimson", ls=":", lw=1.5)
plt.text(cross*1.1, 1e12, f"kesişim\nn = {cross:.0f}", fontsize=9, color="crimson")
plt.xlabel("dizi uzunluğu n"); plt.ylabel("ileri geçiş başına FLOP")
plt.title(f"Dikkat ancak uzun dizilerde baskın olur (d={d}, L={L})")
plt.legend(fontsize=9); plt.grid(alpha=0.3, which="both"); plt.tight_layout(); plt.show()


## 5. Mixture of Experts: Parametreleri FLOP'lardan Ayırmak

Parametrelerin çoğu FFN'de olduğuna göre, tek bir FFN'yi $E$ tane uzman FFN ile değiştirip her
**token**'ı yalnızca en iyi $k$ tanesine yönlendirebiliriz. Toplam parametre yaklaşık $E$ katına,
token başına aktif FLOP ise yaklaşık $k$ katına çıkar. $E = 64$, $k = 2$ ile bir model aynı hesap
bütçesiyle $32\times$ daha fazla bilgi taşıyabilir.

Zor kısım yönlendirme dağılımıdır. Yalnızca görev kaybıyla eğitilen bir softmax yönlendirici çöker:
birkaç uzman erken kazanır, tüm gradyanı alır ve daha da iyileşir. Standart çözüm bir yardımcı
**yük dengeleme kaybıdır**:

$$\mathcal{L}_{\text{aux}} = E \sum_{i=1}^{E} f_i \, P_i,$$

burada $f_i$, $i$ numaralı uzmana yönlendirilen token oranı, $P_i$ ise onun ortalama yönlendirici
olasılığıdır. Çarpım, her ikisi de tekdüze olduğunda minimumdur; $P_i$ türevlenebilir olduğu için
gradyan olasılık kütlesini popüler uzmanlardan uzaklaştırır.


In [ ]:
def route(logits, k=2):
    idx = np.argsort(-logits, axis=1)[:, :k]
    p   = np.exp(logits - logits.max(1, keepdims=True))
    p  /= p.sum(1, keepdims=True)
    return idx, p

def simulate(n_steps=600, E=16, k=2, tokens=512, aux_weight=0.0, lr=0.05, seed=0):
    rng = np.random.default_rng(seed)
    bias = rng.normal(0, 0.05, E)                    # yönlendirici tercihleri (öğrenilen kısım)
    hist = []
    for _ in range(n_steps):
        logits = bias + rng.normal(0, 0.5, (tokens, E))
        idx, p = route(logits, k)
        f = np.bincount(idx.ravel(), minlength=E) / (tokens*k)
        P = p.mean(0)
        hist.append(f.copy())
        # "görev" gradyanı: kullanılan uzmanları ödüllendirir  ->  zengin daha zengin olur
        bias += lr * (f - 1.0/E)
        # yardımcı yük dengeleme gradyanı: kütleyi aşırı kullanılan uzmanlardan uzaklaştırır
        bias -= aux_weight * lr * E * f
    return np.array(hist)

h_no  = simulate(aux_weight=0.0)
h_aux = simulate(aux_weight=0.1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
for ax, h, name in zip(axes[:2], [h_no, h_aux], ["Dengeleme kaybı yok", "Dengeleme kaybı ile"]):
    ax.stackplot(np.arange(h.shape[0]), h.T, alpha=0.85)
    ax.axhline(1/h.shape[1], ls="--", c="k", lw=1)
    ax.set_xlabel("adım"); ax.set_ylabel("uzman başına token oranı")
    ax.set_title(name); ax.set_ylim(0, 1)

# Parametre sayısına karşı aktif FLOP
Es = np.array([1, 2, 4, 8, 16, 32, 64, 128])
axes[2].loglog(Es, Es, "o-", lw=2, label="toplam parametre (göreli)")
axes[2].loglog(Es, np.full_like(Es, 2, dtype=float), "s-", lw=2, label="token başına aktif FLOP (k=2)")
axes[2].set_xlabel("uzman sayısı E"); axes[2].set_ylabel("göreli maliyet")
axes[2].set_title("MoE kapasiteyi hesaptan ayırır")
axes[2].legend(fontsize=9); axes[2].grid(alpha=0.3, which="both")

plt.tight_layout(); plt.show()

for name, h in [("yardımcı kayıp yok", h_no), ("yardımcı kayıp var", h_aux)]:
    f = h[-1]
    print(f"{name:20s}: en yoğun uzman token'ların {f.max():5.1%} kadarını alıyor "
          f"(tekdüze = {1/f.size:.1%}), kullanılmayan uzman = {(f < 0.001).sum()}/{f.size}")


## 6. Tümevarımsal Önyargıya Karşı Ölçek: ViT ve ESA

Bir ESA yerelliği, öteleme eşdeğerliğini ve ağırlık paylaşımını mimariye gömer. Vision Transformer
ise neredeyse hiçbir şey gömmez: görüntüyü yamalara böler ve yapıyı dikkat mekanizmasının keşfetmesine
bırakır. Sonuç, veri verimliliği eğrilerinin **kesişmesidir** — ESA'nın önselleri az veri rejiminde
bir armağan, çok veri rejiminde bir kısıttır.

Kaba bir model: güçlü önselleri olan bir mimari daha düşük hatayla başlar ama daha küçük bir üsle
iyileşir; çünkü hipotez uzayının bir kısmı ne kadar veri görürse görsün ona kapalıdır.


In [ ]:
D = np.logspace(3, 9, 300)
err_cnn = 0.10 + 4.0*D**(-0.28)          # güçlü önsel: iyi başlangıç, düşük eğim, yüksek taban
err_vit = 0.045 + 90.0*D**(-0.42)        # zayıf önsel: kötü başlangıç, dik eğim, düşük taban

cross = D[np.argmin(np.abs(err_cnn - err_vit))]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
axes[0].loglog(D, err_cnn, lw=2, label="ESA (güçlü tümevarımsal önyargı)")
axes[0].loglog(D, err_vit, lw=2, label="ViT (zayıf tümevarımsal önyargı)")
axes[0].axvline(cross, ls=":", c="crimson", lw=1.5)
axes[0].text(cross*1.15, 0.5, f"kesişim\nD = {cross:.1e}", fontsize=9, c="crimson")
axes[0].set_xlabel("eğitim kümesi boyutu D"); axes[0].set_ylabel("hata")
axes[0].set_title("Önseller veri azken yardım eder, çokken tavan koyar")
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3, which="both")

# Yamalama: bir ViT görüntüyü nasıl diziye çevirir
img = np.add.outer(np.sin(np.linspace(0, 6, 32)), np.cos(np.linspace(0, 6, 32)))
P = 8
patches = img.reshape(32//P, P, 32//P, P).transpose(0, 2, 1, 3).reshape(-1, P*P)
axes[1].imshow(patches, aspect="auto", cmap="RdBu_r")
axes[1].set_xlabel("yama içindeki düzleştirilmiş pikseller (P*P = 64)")
axes[1].set_ylabel("yama indeksi (dizi konumu)")
axes[1].set_title(f"32x32 görüntü -> {patches.shape[1]} boyutlu {patches.shape[0]} token")

plt.tight_layout(); plt.show()
print(f"Yama boyutu {P} olan bir ViT, 32x32 görüntüyü {patches.shape[0]} uzunluğunda bir diziye çevirir.")
print("Yama boyutunu yarıya indirmek dizi uzunluğunu dörde katlar -> dikkat maliyeti 4 katına çıkar.")


## 7. Özet

| Kavram | Açıklama |
|---|---|
| **Kuvvet yasası ölçeklemesi** | $L = E + (N_c/N)^{\alpha}$; küçük üsler ilerlemenin pahalı olduğu anlamına gelir |
| **İndirgenemez kayıp $E$** | Verinin entropisi; uydurmadan önce çıkarılmalıdır |
| **$C \approx 6ND$** | Transformer eğitimi için standart FLOP kestirimi |
| **Hesap-optimal** | $N^\star \propto C^{\beta/(\alpha+\beta)}$, $D^\star \propto C^{\alpha/(\alpha+\beta)}$; kabaca eşit büyüme |
| **Çıkarım maliyeti** | Modelleri hesap-optimal token sayısının ötesinde eğitmeyi haklı çıkarır |
| **Ortaya çıkış** | Çoğu zaman süreksiz metriklerin bir yansımasıdır; sürekli metrikler de raporlanmalı |
| **Parametre bütçesi** | $\approx 12d^2L$; bunun yaklaşık üçte ikisi FFN'dedir |
| **Mixture of Experts** | Parametre $\times E$, FLOP $\times k$; yük dengeleme kaybı şarttır |
| **Tümevarımsal önyargı** | Güçlü önseller küçük $D$'de, zayıf önseller büyük $D$'de kazanır |

**Sonraki Defter →** Verimli Dikkat ve Uzun Bağlam
